In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Clean data") \
                            .master("local[*]") \
                            .config("spark.driver.memory" , "8g") \
                            .config("spark.executor.memory" , "8g") \
                            .config("spark.sql.adaptive.enabled" , "true") \
                            .config("spark.serializer" , "org.apache.spark.serializer.KryoSerializer") \
                            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 15:46:26 WARN Utils: Your hostname, Admin-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/02 15:46:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 15:46:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", "file:///")

In [10]:
df = spark.read.parquet("data/part-1774432835370.parquet")

In [11]:
df.show()

+-----------------+--------------------+-------------+--------+-----------------+--------------------+--------------------+-------------+---------------+--------------------+--------------------+------------------+--------------------+-----------+------------+--------------------+
|              key|               title|edition_count|cover_id|cover_edition_key|             subject|       ia_collection|printdisabled|lending_edition|  lending_identifier|             authors|first_publish_year|                  ia|public_scan|has_fulltext|        availability|
+-----------------+--------------------+-------------+--------+-----------------+--------------------+--------------------+-------------+---------------+--------------------+--------------------+------------------+--------------------+-----------+------------+--------------------+
|  /works/OL21177W|   Wuthering Heights|         2886|12818862|      OL38586477M|["form:novel", "g...|["365-Books-by-Wo...|         true|    OL57648863M|w

In [12]:
df.coalesce(1).write.mode("overwrite").parquet("test")

In [ ]:
from pyspark.sql.functions import * 
import boto3 
from functools import reduce 

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
list_file = []
list_col = []
for file in files :
    df = spark.read.format("parquet").load(file) 
    
    df = df.withColumn("cover_id" , col("cover_id").cast("long"))
    for c in df.columns :
        if c not in list_col :
            list_col.append(c)
    for c in list_col :
        if c not in df.columns :
            df = df.withColumn(c , lit(None).cast("string"))
    list_file.append(df)
df = reduce(lambda a,b : a.union(b) , list_file)

df.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/works/")